# 02 — Sequential Pattern Mining

This notebook mines frequent symbolic sequences using PrefixSpan.

Input:
- `positive_sequences.pkl`
- `negative_sequences.pkl`

Output:
- Frequent positive patterns
- Frequent negative patterns

The notebook does not perform classification or sequence-distance modeling.


In [ ]:
from pathlib import Path
import pickle
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
SEQUENCE_DIR = PROJECT_ROOT / "outputs" / "sequences"
PATTERN_DIR = PROJECT_ROOT / "outputs" / "patterns"
PATTERN_DIR.mkdir(parents=True, exist_ok=True)

MIN_SUPPORT = 0.05
MIN_PATTERN_LENGTH = 2
MAX_PATTERN_LENGTH = 5

with open(SEQUENCE_DIR / "positive_sequences.pkl", "rb") as f:
    positive_sequences = pickle.load(f)

with open(SEQUENCE_DIR / "negative_sequences.pkl", "rb") as f:
    negative_sequences = pickle.load(f)

print("Positive sequences:", len(positive_sequences))
print("Negative sequences:", len(negative_sequences))


## PrefixSpan representation

Each patient sequence is represented as a list of hourly event-items:

```text
[
    ["HR_HIGH", "MAP_NORMAL"],
    ["HR_HIGH", "MAP_LOW"],
    ["TEMP_HIGH"]
]
```

PrefixSpan should discover ordered subsequences while preserving temporal order.


In [ ]:
# Try to import a PrefixSpan implementation.
try:
    from prefixspan import PrefixSpan
except ImportError as exc:
    raise ImportError(
        "Install the 'prefixspan' package before running this notebook."
    ) from exc


def mine_prefixspan(sequences, min_support=0.05,
                    min_length=2, max_length=5):
    if not sequences:
        return []

    min_count = max(1, int(len(sequences) * min_support))

    # PrefixSpan implementations commonly expect each item to be hashable.
    # Flatten each hourly itemset into tokens while preserving a separator
    # convention through tuple tokens.
    transformed = []
    for sequence in sequences:
        tokens = []
        for hour_idx, itemset in enumerate(sequence):
            for item in itemset:
                tokens.append((hour_idx, item))
        transformed.append(tokens)

    ps = PrefixSpan(transformed)
    raw = ps.frequent(min_count)

    results = []
    for support_count, pattern in raw:
        items = [item for _, item in pattern]
        if min_length <= len(items) <= max_length:
            results.append({
                "pattern": tuple(items),
                "support_count": support_count,
                "support": support_count / len(sequences),
                "length": len(items),
            })

    return results


In [ ]:
positive_patterns = mine_prefixspan(
    positive_sequences,
    MIN_SUPPORT,
    MIN_PATTERN_LENGTH,
    MAX_PATTERN_LENGTH
)

negative_patterns = mine_prefixspan(
    negative_sequences,
    MIN_SUPPORT,
    MIN_PATTERN_LENGTH,
    MAX_PATTERN_LENGTH
)

positive_df = pd.DataFrame(positive_patterns)
negative_df = pd.DataFrame(negative_patterns)

positive_df.to_csv(PATTERN_DIR / "frequent_positive_patterns.csv", index=False)
negative_df.to_csv(PATTERN_DIR / "frequent_negative_patterns.csv", index=False)

print("Positive patterns:", len(positive_df))
print("Negative patterns:", len(negative_df))


In [ ]:
display(positive_df.sort_values("support", ascending=False).head(20))
